# Component 2 - Stat Predictions

Predicts each player's points, rebounds, assists and threes for each game, using `scripts/stat_model.py` on the feature file (run *Player Stats Ingestion* and *Feature Engineering* first). No odds yet. Takes about 2 minutes.

**Data:** last season only, rows where the player has 10+ earlier games.

**Split (by date):** train on the first half of the season, test on the second half. Settings for each model are picked on the last 20% of the train half, then the model is refit on the whole first half. The test half is never seen while fitting.

**Scores (test half):**
- **MAE**: average miss, in the stat's units (e.g. 4.5 points). Lower is better.
- **R2**: share of game-to-game variation explained. Higher is better.
- **Bias**: average prediction minus actual. Near 0 means no lean high or low.

In [ ]:
LEAGUE = "nba"

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "stat_model.py").exists())
sys.path.insert(0, str(root / "scripts"))

import importlib
import fetch_player_stats, build_features, stat_model as sm
for m in (fetch_player_stats, build_features, sm):
    importlib.reload(m)  # pick up script edits without restarting the kernel

import pandas as pd
pd.set_option("display.max_columns", None)

## Train and test

In [ ]:
out = sm.run(LEAGUE)

## Scores on the test half

Baselines (last-10 and season averages) are what any model has to beat.

In [ ]:
for metric in ("MAE", "R2", "Bias"):
    print(metric)
    display(sm.score_table(out["results"], metric).round(3))

Misses grow with minutes: players who play more score more, so their errors are bigger in raw units.

In [ ]:
sm.by_minutes(out["preds"]).round(3)

## What drives the predictions

Permutation importance for the gradient boosting model: how much test MAE rises when one feature is shuffled.

In [ ]:
for stat in ("pts", "reb", "ast", "fg3m"):
    print(sm.STAT_NAMES[stat])
    display(sm.importance(out, stat).round(3).to_frame())

## Predictions, player by player

One row per player-game in the test half. `{stat}` is the actual, `{stat}_pred` is the ensemble, and `{stat}_pred_{model}` is each model.

In [ ]:
preds = out["preds"]
cols = ["game_date", "player_name", "team", "opponent", "pts", "pts_pred", "reb", "reb_pred", "ast", "ast_pred", "fg3m", "fg3m_pred"]
preds[cols].round(1).head(20)

In [ ]:
PLAYER = "Josh Giddey"
preds.loc[preds["player_name"] == PLAYER, cols].round(1)

## Save

In [ ]:
sm.save(LEAGUE, out)